# Quick checks: `ods_alpha.scd1_mrc_pos_rent`

Отдельная тетрадка для 4 проверок:
1. Есть ли доступ к таблице.
2. Дубли по ключу `c_nmrc + d_rent`.
3. Конфликты, где у одного `c_nmrc + d_rent` разные `n_amt`.
4. Проверка за апрель для кейса `agr_id=413636181589`, `inn=2259000869`.

In [ ]:
import re
import time
from decimal import Decimal, InvalidOperation

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None

In [ ]:
# === Таблицы ===
mrc_table = 'ods_alpha.scd1_mrc_pos_rent'
agr_terms_table = 'ods_alpha.scd1_agr_terms'
agreements_table = 'ods_alpha.scd1_agreements'
companies_table = 'ods_alpha.scd1_companies'

# === Параметры кейса ===
month_start = '2026-04-01'
month_end = '2026-04-30'
target_agr_id = normalize_agr_q1('413636181589')
target_inn = normalize_inn_q1('2259000869')

# === Подключение ===
impala_db = 'sandbox_ai'
impala_mem_limit = '8g'
impala_user_name = 'Shestopalov-VYur'

print('check month:', month_start, '..', month_end)
print('target agr_id:', target_agr_id)
print('target inn:', target_inn)
print('table:', mrc_table)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user_name}
)
imp._init_connection()


def run_sql(sql_text, step_name='query', mem_limit=impala_mem_limit):
    start_ts = time.perf_counter()
    print(f'[{step_name}] start')
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        df = imp.fetch(sql_text)
    elapsed = round(time.perf_counter() - start_ts, 2)
    rows = len(df) if isinstance(df, pd.DataFrame) else 0
    print(f'[{step_name}] done in {elapsed}s, rows={rows:,}')
    return df


print('Impala connection initialized')

## 1) Проверка доступа к таблице

In [ ]:
sql_access = f"""
select 1 as probe_ok
from {mrc_table}
limit 1
"""

access_ok = False
access_error = None

try:
    access_df = run_sql(sql_access, step_name='access_check')
    access_ok = True
    print('ACCESS_OK: есть SELECT-доступ к таблице')
    display(access_df)
except Exception as exc:
    access_error = f'{type(exc).__name__}: {exc}'
    print('ACCESS_ERROR:', access_error)

access_result_df = pd.DataFrame([
    {'table': mrc_table, 'access_ok': access_ok, 'error': access_error}
])
display(access_result_df)

## 2) Дубли по `c_nmrc + d_rent`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    sql_dup_summary = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), grouped as (
        select
            c_nmrc,
            d_rent_dt,
            count(*) as row_cnt
        from active
        group by c_nmrc, d_rent_dt
    )
    select
        (select count(*) from active) as active_rows,
        (select count(*) from grouped) as key_cnt,
        coalesce((select count(*) from grouped where row_cnt > 1), 0) as duplicate_key_cnt,
        coalesce((select sum(row_cnt - 1) from grouped where row_cnt > 1), 0) as duplicate_row_overhead
    """

    dup_summary_df = run_sql(sql_dup_summary, step_name='dup_summary')
    if len(dup_summary_df):
        key_cnt = float(dup_summary_df.loc[0, 'key_cnt'])
        dup_key_cnt = float(dup_summary_df.loc[0, 'duplicate_key_cnt'])
        dup_summary_df['duplicate_key_pct'] = (dup_key_cnt / key_cnt * 100.0) if key_cnt else np.nan
    display(dup_summary_df)

    sql_dup_top = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    )
    select
        c_nmrc,
        d_rent_dt,
        count(*) as row_cnt
    from active
    group by c_nmrc, d_rent_dt
    having count(*) > 1
    order by row_cnt desc, d_rent_dt desc, c_nmrc
    limit 200
    """

    dup_top_df = run_sql(sql_dup_top, step_name='dup_top_200')
    display(dup_top_df)

## 3) Конфликты: один `c_nmrc + d_rent`, разные `n_amt`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    sql_conflict_summary = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), grouped as (
        select
            c_nmrc,
            d_rent_dt,
            count(*) as row_cnt,
            count(distinct n_amt_num) as distinct_amt_cnt
        from active
        group by c_nmrc, d_rent_dt
    )
    select
        (select count(*) from grouped) as key_cnt,
        coalesce((select count(*) from grouped where distinct_amt_cnt > 1), 0) as conflict_key_cnt,
        coalesce((select sum(row_cnt) from grouped where distinct_amt_cnt > 1), 0) as rows_in_conflicts
    """

    conflict_summary_df = run_sql(sql_conflict_summary, step_name='conflict_summary')
    if len(conflict_summary_df):
        key_cnt = float(conflict_summary_df.loc[0, 'key_cnt'])
        conflict_cnt = float(conflict_summary_df.loc[0, 'conflict_key_cnt'])
        conflict_summary_df['conflict_key_pct'] = (conflict_cnt / key_cnt * 100.0) if key_cnt else np.nan
    display(conflict_summary_df)

    sql_conflict_top = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    )
    select
        c_nmrc,
        d_rent_dt,
        count(*) as row_cnt,
        count(distinct n_amt_num) as distinct_amt_cnt,
        min(n_amt_num) as min_amt,
        max(n_amt_num) as max_amt
    from active
    group by c_nmrc, d_rent_dt
    having count(distinct n_amt_num) > 1
    order by distinct_amt_cnt desc, row_cnt desc, d_rent_dt desc, c_nmrc
    limit 200
    """

    conflict_top_df = run_sql(sql_conflict_top, step_name='conflict_top_200')
    display(conflict_top_df)

    sql_conflict_rows = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            cast(ods_commit_ts as timestamp) as ods_commit_ts,
            cast(ods_insert_ts as timestamp) as ods_insert_ts,
            cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
            cast(ods_op_type as string) as ods_op_type,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), conflict_keys as (
        select c_nmrc, d_rent_dt
        from active
        group by c_nmrc, d_rent_dt
        having count(distinct n_amt_num) > 1
    )
    select a.*
    from active a
    join conflict_keys k
      on k.c_nmrc = a.c_nmrc
     and k.d_rent_dt = a.d_rent_dt
    order by a.d_rent_dt desc, a.c_nmrc, a.n_amt_num desc, a.ods_commit_ts desc
    limit 500
    """

    conflict_rows_df = run_sql(sql_conflict_rows, step_name='conflict_rows_500')
    display(conflict_rows_df)

## 4) Кейс за апрель: `agr_id=413636181589`, `inn=2259000869`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    target_agr_sql = str(target_agr_id).replace("'", "''")
    target_inn_sql = str(target_inn).replace("'", "''")

    base_case_cte = f"""
    with target_agreements as (
        select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.abs_agr_id as string) as agr_id,
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
        from {agreements_table} a
        join {companies_table} c
          on cast(c.n_cmp as string) = cast(a.n_cmp_client as string)
        where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
          and coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
          and cast(a.abs_agr_id as string) = '{target_agr_sql}'
          and regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{target_inn_sql}'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    ), target_terms as (
        select distinct
            ta.n_agr,
            cast(t.c_nmrc as string) as c_nmrc
        from target_agreements ta
        join {agr_terms_table} t
          on cast(t.n_agr as string) = ta.n_agr
         and coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
         and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
         and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
         and t.c_nmrc is not null
    ), rent_base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            cast(ods_commit_ts as timestamp) as ods_commit_ts,
            cast(ods_insert_ts as timestamp) as ods_insert_ts,
            cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
          and c_nmrc is not null
    ), rent_ranked as (
        select
            *,
            row_number() over (
                partition by c_nmrc, d_rent_dt
                order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
            ) as rn
        from rent_base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), rent_dedup as (
        select c_nmrc, d_rent_dt, n_amt_num
        from rent_ranked
        where rn = 1
    )
    """

    sql_case_summary = f"""
    {base_case_cte}
    select
        (select count(*) from target_agreements) as active_agreements_cnt,
        (select count(distinct n_agr) from target_agreements) as n_agr_cnt,
        (select count(distinct c_nmrc) from target_terms) as nmrc_from_terms_cnt,
        coalesce((select count(*) from rent_dedup r join target_terms t on t.c_nmrc = r.c_nmrc), 0) as rent_rows_cnt,
        coalesce((select sum(r.n_amt_num) from rent_dedup r join target_terms t on t.c_nmrc = r.c_nmrc), 0) as commission_monthly_april
    """

    case_summary_df = run_sql(sql_case_summary, step_name='case_summary_april')
    display(case_summary_df)

    sql_case_nmrc = f"""
    {base_case_cte}
    select
        ta.agr_id,
        ta.inn,
        tt.n_agr,
        tt.c_nmrc
    from target_terms tt
    join target_agreements ta
      on ta.n_agr = tt.n_agr
    order by tt.c_nmrc
    """

    case_nmrc_df = run_sql(sql_case_nmrc, step_name='case_nmrc_list')
    display(case_nmrc_df)

    sql_case_rent_details = f"""
    {base_case_cte}
    select
        ta.agr_id,
        ta.inn,
        tt.n_agr,
        r.c_nmrc,
        r.d_rent_dt,
        r.n_amt_num
    from rent_dedup r
    join target_terms tt
      on tt.c_nmrc = r.c_nmrc
    join target_agreements ta
      on ta.n_agr = tt.n_agr
    order by r.d_rent_dt, r.c_nmrc
    """

    case_rent_details_df = run_sql(sql_case_rent_details, step_name='case_rent_details_april')
    display(case_rent_details_df)

## Как читать результат

- **Пункт 1:** `access_ok=True` значит доступ есть.
- **Пункт 2:** `duplicate_key_cnt > 0` значит есть дубли по `c_nmrc + d_rent`.
- **Пункт 3:** `conflict_key_cnt > 0` значит есть конфликтные суммы `n_amt` по одному ключу.
- **Пункт 4:**
  - `nmrc_from_terms_cnt` показывает, найдена ли связка `agr_id+inn -> c_nmrc` за апрель;
  - `commission_monthly_april` — сумма `n_amt` за апрель после дедупликации;
  - детальная таблица показывает все строки, вошедшие в сумму.